# Inference-time prompt embedding optimization

Example for *Inference-Time Optimization of Prompt Embeddings for Text-to-Image Diffusion Models Using Evolutionary Algorithms*.

This notebook runs `EmbeddingOptimizer` from `embedding_optimizer.py` on one prompt, using **SDXL Turbo, sep-CMA-ES, and CLIPScore** by default. The diffusion and scorer models stay frozen. It uses a small demonstration budget, not the paper's full evaluation protocol; see [README.md](README.md) for tuning and DrawBench reproduction.

Activate the `evo_embs` Conda environment from `environment.yml` and select its Python kernel. Start Jupyter from the repository root and run cells in order. Model checkpoints download on first use. Changing optimizers or objectives requires rerunning configuration and initialization.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "embedding_optimizer.py").is_file():
    raise RuntimeError("Start Jupyter from the repository root containing embedding_optimizer.py.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## Dependencies

Use the repository's shared dependencies rather than a separate notebook-specific list. If the current kernel needs installation, uncomment and run the following cell, then restart the kernel and rerun from the top.

In [ ]:
# %pip install -r requirements.txt

## Configure a single-prompt run

Load the current single-prompt YAML and override it with a small, explicit example. `optimization_target` supports `prompt_embeddings` and `latent_noise`; this example optimizes prompt embeddings.

Choose `cmaes` (with `cmaes_variant: sep`), `snes`, `cosyne`, `ga`, `zero_order`, `adam` (AdamW), or `random_sampler`. For a paper-style objective, set exactly one of `clip_score_weight`, `image_reward_score_weight`, and `hpsv2_score_weight` to 1 and the others to 0. Keep the corresponding scorer configured. The default scores CLIP only and skips zero-weight metrics.

For SD 1.5 use 50 denoising steps and guidance 7.5; SDXL Turbo uses 1 step and guidance 0. SANA-Sprint uses 2 steps and guidance 4.5, with its exact checkpoint and resolution set explicitly. These model choices can require different GPU memory capacities.

In [ ]:
import yaml

config_path = REPO_ROOT / "algorithms/config/config_run_single_prompt.yaml"
with config_path.open(encoding="utf-8") as handle:
    experimental_setup_parameters = yaml.safe_load(handle)

experimental_setup_parameters.update({
    "selected_prompt": "an astronaut riding a horse on Mars",
    "optimization_method": "cmaes",
    "optimization_target": "prompt_embeddings",
    "seed": 42,
    "cuda": 0,  # Use "cpu" for CPU execution, which is much slower.
    "model_id": "stabilityai/sdxl-turbo",
    "model_backend": "sdxl",
    "base_model_id": None,
    "unet_model_id": None,
    "num_inference_steps": 1,
    "guidance_scale": 0.0,
    "height": 512,
    "width": 512,
    "torch_dtype": "float32",
    "batch_size": 4,
    "aesthetic_predictor": 2,
    "aesthetic_score_weight": 0.0,
    "clip_score_weight": 1.0,
    "image_reward_score_weight": 0.0,
    "hpsv2_score_weight": 0.0,
    "pickscore_score_weight": 0.0,
    "max_clip_score": 1.0,
    "clip_model_name": "ViT-B/32",
    "evaluate_zero_weight_metrics": False,
    "zero_weight_metrics": [],
    "results_folder": "results_notebook",
    "resume_experiment": False,
    "save_gens": True,
    "time_limit_seconds": None,
    # Small demo budgets; set every method-specific override explicitly.
    "num_generations": 3,
    "pop_size": 8,
    "cmaes_variant": "sep",
    "sigma": 0.1,
    "snes_num_generations": 3,
    "snes_pop_size": 8,
    "snes_sigma": 0.1,
    "snes_eta_mu": 1.0,
    "snes_eta_sigma": None,
    "cosyne_num_generations": 3,
    "cosyne_pop_size": 8,
    "cosyne_init_range": 0.1,
    "cosyne_mutation_probability": 0.3,
    "cosyne_mutation_scale": 0.1,
    "cosyne_parent_count": 4,
    "cosyne_offspring_count": 4,
    "zero_order_num_generations": 3,
    "zero_order_pop_size": 8,
    "zero_order_sigma": 0.1,
    "num_iterations": 10,
    "adam_lr": 1e-3,
    "adam_weight_decay": 0.0,
    "adam_eps": 1e-8,
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_max_grad_norm": None,
    "num_images_to_generate": 24,
})

print(yaml.safe_dump(experimental_setup_parameters, sort_keys=False))

## Run optimization

Initialization loads the selected diffusion pipeline and active scorers. The run writes images, CSV metrics, plots, the effective configuration, and checkpoints to the returned folder. Rerunning with the same settings may reuse the same output path; choose a new `results_folder` to keep separate experiments.

In [ ]:
from embedding_optimizer import EmbeddingOptimizer

optimizer_engine = EmbeddingOptimizer(experimental_setup_parameters)

method_runners = {
    "cmaes": optimizer_engine.run_cmaes_optimization,
    "snes": optimizer_engine.run_snes_optimization,
    "cosyne": optimizer_engine.run_cosyne_optimization,
    "ga": optimizer_engine.run_ga_optimization,
    "zero_order": optimizer_engine.run_zero_order_optimization,
    "adam": optimizer_engine.run_adam_optimization,
    "random_sampler": optimizer_engine.run_random_sampler_optimization,
}
method = experimental_setup_parameters["optimization_method"]
if method not in method_runners:
    raise ValueError(f"Unsupported optimization method: {method}")
results_folder = Path(method_runners[method]())
print(f"Results saved to: {results_folder.resolve()}")

## Inspect images and scores

Saved images use JPEG filenames. CSVs report canonical scores from saved images; these may differ slightly from the tensor scores used during optimization. The table includes the initial baseline and recorded optimization steps. Scores for skipped zero-weight metrics are not independent evaluations.

In [ ]:
from IPython.display import Image as IPythonImage, display

for label, filename in [("Initial image", "it_0.jpg"), ("Best image", "best_all.jpg")]:
    print(label)
    display(IPythonImage(filename=str(results_folder / filename)))

In [ ]:
import pandas as pd

csv_name = "score_results.csv" if method == "adam" else "fitness_results.csv"
results = pd.read_csv(results_folder / csv_name)
display(results)

In [ ]:
# Show the objective plot and plots for the active guiding metrics.
plot_names = ["loss_evolution.jpg" if method == "adam" else "fitness_evolution.jpg"]
metric_plots = {
    "clip_score_weight": "clip_evolution.jpg" if method == "adam" else "clip_score_evolution.jpg",
    "image_reward_score_weight": "image_reward_evolution.jpg",
    "hpsv2_score_weight": "hpsv2_evolution.jpg",
    "aesthetic_score_weight": "aesthetic_evolution.jpg" if method == "adam" else "aesthetic_score_evolution.jpg",
    "pickscore_score_weight": "pickscore_evolution.jpg",
}
plot_names.extend(filename for weight, filename in metric_plots.items()
                  if experimental_setup_parameters[weight] != 0)
for filename in plot_names:
    plot_path = results_folder / filename
    if plot_path.is_file():
        print(filename)
        display(IPythonImage(filename=str(plot_path)))
    else:
        print(f"Plot not produced by this optimizer: {filename}; inspect {csv_name} instead.")